# `find_rotation_axis()`

`nematics3d.find_rotation_axis()` fits one oriented rotation axis to an ordered sequence of three-dimensional unit directors and reports diagnostics for judging the fit.

## What `find_rotation_axis()` is for

Use this function when a sequence of directors is expected to rotate approximately in one plane. The function finds the axis normal to that plane. It also uses the order of the sequence to orient the axis: reversing a consistently rotating sequence reverses the returned axis.

## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** It imports `NumPy` and the public `nematics3d` package used in the examples.

In [1]:
import numpy as np
import nematics3d as n3d

## Minimal example

The following directors rotate counterclockwise through a quarter circle in the $xy$-plane. They are ordered samples and every row is a unit vector. The expected oriented axis is therefore the positive $z$-axis.

In [2]:
angles = np.linspace(0.0, np.pi / 2.0, 9)
directors = np.column_stack(
    [
        np.cos(angles),
        np.sin(angles),
        np.zeros_like(angles),
    ]
)

result = n3d.find_rotation_axis(directors)
result.axis

array([0., 0., 1.])

### What is returned

The function returns a `RotationAxisResult`, not a bare array. `result.axis` is the fitted unit vector. The other fields describe how closely the data match a common rotation plane and how consistently the sequence rotates in one direction.

In [3]:
print(result)
print("diagnostics as a dictionary:", result.metric)

RotationAxisResult: director rotation-axis fit
  axis                 = [0., 0., 1.],
  orthogonality_score  = 1,
  rms_sin_theta        = 0,
  tilt_angle_degrees   = 0,
  rotation_consistency = 1,
  eigenvalues          = [0.    , 1.9863, 7.0137],
diagnostics as a dictionary: {'orthogonality_score': 1.0, 'rms_sin_theta': 0.0, 'tilt_angle_degrees': 0.0, 'rotation_consistency': 1.0, 'eigenvalues': array([0.        , 1.98633025, 7.01366975])}


### Reading the output

For this exact planar rotation, `axis` is approximately `[0, 0, 1]`. `orthogonality_score` and `rotation_consistency` are approximately 1, while `rms_sin_theta` and `tilt_angle_degrees` are approximately 0. Together these values mean that the directors lie in a plane normal to the fitted axis and advance consistently in one orientation.

Do not interpret one diagnostic in isolation. A high `orthogonality_score` indicates a well-defined common plane, whereas a high `rotation_consistency` indicates consistent ordered motion within that plane. Neither score alone establishes that the input represents the physical structure you intended to sample.

## Arguments

```python
find_rotation_axis(directors) -> RotationAxisResult
```

| Argument | Accepted value | Meaning |
| --- | --- | --- |
| `directors` | Array-like with shape `(N, 3)`, where `N >= 2` | An ordered sequence of finite, normalized three-dimensional directors. Row order determines the orientation of the returned axis. |

The function validates the shape, finiteness, number of rows, and unit length of every director. It does not normalize the input or repair arbitrary director sign flips.

## Special examples

### Reversing the ordered rotation

The best-fit geometric line is unchanged when the samples are reversed, but its orientation changes.

In [4]:
reverse_result = n3d.find_rotation_axis(directors[::-1])

print("forward axis:", result.axis)
print("reverse axis:", reverse_result.axis)
print("axes are oppositely oriented:", np.allclose(result.axis, -reverse_result.axis))

forward axis: [0. 0. 1.]
reverse axis: [-0. -0. -1.]
axes are oppositely oriented: True


### A sequence tilted out of its best-fit plane

The next sequence retains a clear rotation around the $z$-axis but has a constant $z$ component. Its nonzero tilt diagnostics quantify the departure from a plane perpendicular to the fitted axis.

In [5]:
angles_tilted = np.linspace(0.0, 2.0 * np.pi, 20, endpoint=False)
z_component = 0.15
xy_component = np.sqrt(1.0 - z_component**2)
directors_tilted = np.column_stack(
    [
        xy_component * np.cos(angles_tilted),
        xy_component * np.sin(angles_tilted),
        np.full_like(angles_tilted, z_component),
    ]
)

tilted_result = n3d.find_rotation_axis(directors_tilted)
print("axis:", tilted_result.axis)
print("orthogonality score:", tilted_result.orthogonality_score)
print("tilt angle (degrees):", tilted_result.tilt_angle_degrees)
print("rotation consistency:", tilted_result.rotation_consistency)

axis: [-0.0000000e+00  1.5162785e-16  1.0000000e+00]
orthogonality score: 0.9775
tilt angle (degrees): 8.626926558678633
rotation consistency: 1.0


## Returned object

### `RotationAxisResult`

| Field | Interpretation |
| --- | --- |
| `axis` | Best-fit unit axis, oriented by the net ordered rotation. |
| `orthogonality_score` | $1-\lambda_{\min}/\sum_i\lambda_i$. Values near 1 mean that the directors lie close to a plane normal to `axis`. |
| `rms_sin_theta` | Root-mean-square component of the normalized directors along `axis`. Smaller values indicate a more planar fit. |
| `tilt_angle_degrees` | `arcsin(rms_sin_theta)` expressed in degrees. Smaller values indicate a more planar fit. |
| `rotation_consistency` | Magnitude of net signed rotation divided by total absolute signed rotation. Values near 1 indicate one consistent orientation; values near 0 indicate cancellation or negligible rotation. |
| `eigenvalues` | Three ascending eigenvalues of the director second-moment matrix. The smallest eigenvalue determines the fitted axis. |
| `metric` | A shallow dictionary containing all diagnostic fields except `axis`, convenient for composing higher-level results. |

## Details

### Fitting the common axis

For the director matrix $D$, the function diagonalizes the second-moment matrix $D^T D$. The eigenvector associated with its smallest eigenvalue minimizes the summed squared components of the directors along the axis. It is therefore normal to the best-fit plane containing the directors.

### Orienting the fitted line

An eigenvector defines an unoriented line, so its sign is initially arbitrary. The function sums the cross products of consecutive directors and chooses the sign whose dot product with that sum is nonnegative. This makes the returned axis follow the net ordered rotation when the sequence contains a nonzero directional rotation.

## Possible issues

### The axis sign changes unexpectedly

Check the sample order and director signs. Reversing the order intentionally reverses the axis. Arbitrary sign flips between neighboring headless directors can also change the cross products used for orientation; align the director sequence before fitting.

### The axis sign is not reproducible for a nonrotating sequence

When consecutive cross products cancel or vanish, the data do not select a physical orientation for the best-fit line. In that case `rotation_consistency` is low or zero, and the numerical eigenvector sign should not be interpreted physically.

### The fitted axis is unstable

Inspect `eigenvalues`. If the two smallest eigenvalues are equal or nearly equal, more than one direction fits similarly well and the axis can be sensitive to small perturbations.

### Input validation raises an exception

Supply at least two rows with shape `(N, 3)`, finite entries, and unit norm. Normalize upstream if necessary; `find_rotation_axis()` deliberately rejects non-unit input instead of silently changing it.

## Where `find_rotation_axis()` is used

`QPlanePolar.act_calc_omega()` is the main internal caller. It selects one polar ring, diagonalizes the sampled $Q$ tensors, aligns neighboring directors, and passes the ordered directors to `find_rotation_axis()`. It then stores `rotation_axis.axis` as `OmegaResult.omega` and merges `rotation_axis.metric` with flags describing defects and interpolation-domain coverage.

Call `find_rotation_axis()` directly when the ordered directors have already been prepared and you need the geometric fit independently of a `QPlanePolar` object.

## Useful Links

### Referenced in this tutorial

- [`find_rotation_axis()` source](../../../src/nematics3d/geometry/rotation.py) — defines the fit, orientation rule, validation, and `RotationAxisResult`.
- [`find_rotation_axis()` tests](../../../tests/geometry/test_rotation.py) — records the expected result fields, orientation behavior, and validation errors.
- [`QPlanePolar.act_calc_omega()` source](../../../src/nematics3d/classes/q_plane.py) — shows director alignment and downstream composition into `OmegaResult`.
